<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/projects/romanian-qa-xquad-ro/03_heldout_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Romanian QA — held-out evaluation and Hub release

The two earlier notebooks in this project fine-tuned and evaluated on the **same**
`xquad.ro.json` file, because XQuAD ships no training split. Those numbers measure
fit, not generalization, and should not be compared with published XQuAD results.

This notebook fixes that. It:

1. splits XQuAD-ro **by article**, so no context is shared between train and test,
2. trains both models on the **identical** train split with the identical recipe,
3. selects the checkpoint on a validation split neither model is scored on,
4. reports Exact Match / F1 on a **held-out test split**, across several seeds,
5. optionally pushes both checkpoints to the Hub with model cards.

**Prerequisites** — a GPU runtime (T4 is enough). Roughly 30-45 minutes for the
full matrix of 2 models x 3 seeds.

> **Why not just shuffle the questions?** XQuAD averages ~5 questions per
> paragraph. A question-level split puts the *same passage* on both sides: the
> model reads the context during training, then answers a different question
> about it at test time. Measured on this dataset, **98.3%** of test questions
> would reuse a training context. Cell 3 reproduces that number.

## 1. Setup

In [ ]:
%pip install -q transformers datasets evaluate accelerate

import collections, json, os
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering,
                          DataCollatorWithPadding, get_scheduler, set_seed)
from datasets import Dataset
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: this notebook assumes a GPU. On CPU it will be extremely slow.")

## 2. Build a leak-free split

`data_splits.py` lives next to this notebook. It shuffles **articles** (not
questions) and partitions them, so every paragraph belonging to an article stays
on one side of the split. `verify_no_leakage` asserts that no article, context or
question id appears in two splits — it raises rather than warns.

In [ ]:
# Colab: fetch the helper module that sits beside this notebook in the repo
import urllib.request
if not os.path.exists("data_splits.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pop123-ux/HuggingFace-Project-Learning"
        "/main/projects/romanian-qa-xquad-ro/data_splits.py",
        "data_splits.py",
    )

from data_splits import load_xquad_ro, make_splits, verify_no_leakage

raw = load_xquad_ro()                 # downloads xquad.ro.json from the DeepMind repo
splits = make_splits(raw, train_frac=0.70, val_frac=0.15, seed=42)
summary = verify_no_leakage(splits)   # raises AssertionError on any overlap

for name, counts in summary.items():
    print(f"{name:<12} {counts['articles']:>3} articles  "
          f"{counts['contexts']:>4} contexts  {counts['questions']:>5} questions")

datasets_ = {name: Dataset.from_list(rows) for name, rows in splits.items()}

### Why the split has to be at article level

In [ ]:
# Reproduce the leakage a naive question-level split would introduce.
import random
rows = [r for v in splits.values() for r in v]
random.Random(0).shuffle(rows)
cut = int(len(rows) * 0.7)
train_ctx = {r["context"] for r in rows[:cut]}
held = rows[cut:]
leaked = sum(1 for r in held if r["context"] in train_ctx)
print(f"question-level split: {leaked}/{len(held)} test questions "
      f"({100*leaked/len(held):.1f}%) reuse a context seen during training")
print("article-level split:  0 by construction (asserted above)")

## 3. Tokenization

Extractive QA needs two different tokenizations:

- **training** — sliding windows labelled with `start_positions` / `end_positions`;
  windows that don't contain the answer are labelled at `[CLS]`.
- **evaluation** — the same windows, but keeping `offset_mapping` and the id of
  the example each window came from, so predicted tokens can be mapped back to
  characters and the best answer chosen across all windows of an example.

In [ ]:
MAX_LENGTH, STRIDE = 384, 128

def preprocess_train(examples, tokenizer):
    tok = tokenizer(
        [q.lstrip() for q in examples["question"]],
        examples["context"],
        max_length=MAX_LENGTH, truncation="only_second", stride=STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )
    sample_map = tok.pop("overflow_to_sample_mapping")
    offsets = tok.pop("offset_mapping")
    starts, ends = [], []

    for i, offset in enumerate(offsets):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tok.sequence_ids(i)

        # locate the context portion of this window
        ctx_start = sequence_ids.index(1)
        ctx_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if offset[ctx_start][0] > start_char or offset[ctx_end][1] < end_char:
            starts.append(0); ends.append(0)          # answer not in this window
        else:
            idx = ctx_start
            while idx <= ctx_end and offset[idx][0] <= start_char:
                idx += 1
            starts.append(idx - 1)
            idx = ctx_end
            while idx >= ctx_start and offset[idx][1] >= end_char:
                idx -= 1
            ends.append(idx + 1)

    tok["start_positions"] = starts
    tok["end_positions"] = ends
    return tok


def preprocess_eval(examples, tokenizer):
    tok = tokenizer(
        [q.lstrip() for q in examples["question"]],
        examples["context"],
        max_length=MAX_LENGTH, truncation="only_second", stride=STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )
    sample_map = tok.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(tok["input_ids"])):
        example_ids.append(examples["id"][sample_map[i]])
        sequence_ids = tok.sequence_ids(i)
        # keep offsets only for context tokens; None elsewhere so they can't be predicted
        tok["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tok["offset_mapping"][i])
        ]
    tok["example_id"] = example_ids
    return tok

## 4. Post-processing

The model emits two logit vectors per window. Turning those into an answer means
taking the top-`n_best` start and end candidates, rejecting pairs where the end
precedes the start or the span is too long, and — because one example may span
several windows — choosing the best candidate across all of that example's windows.

In [ ]:
N_BEST, MAX_ANSWER_LENGTH = 20, 30
squad_metric = evaluate.load("squad")

def postprocess(start_logits, end_logits, features, examples):
    per_example = collections.defaultdict(list)
    for idx, feature_id in enumerate(features["example_id"]):
        per_example[feature_id].append(idx)

    predictions = []
    for example in examples:
        context = example["context"]
        best_text, best_score = "", -1e9

        for feature_index in per_example[example["id"]]:
            starts = start_logits[feature_index]
            ends = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            for s in np.argsort(starts)[-1 : -N_BEST - 1 : -1]:
                for e in np.argsort(ends)[-1 : -N_BEST - 1 : -1]:
                    if offsets[s] is None or offsets[e] is None:
                        continue
                    if e < s or e - s + 1 > MAX_ANSWER_LENGTH:
                        continue
                    score = starts[s] + ends[e]
                    if score > best_score:
                        best_score = score
                        best_text = context[offsets[s][0] : offsets[e][1]]

        predictions.append({"id": example["id"], "prediction_text": best_text})

    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return squad_metric.compute(predictions=predictions, references=references)

## 5. Train and evaluate

The recipe is identical for both checkpoints — that is the whole point of the
comparison. The only thing that varies is the pretrained model and the seed.

In [ ]:
NUM_EPOCHS, BATCH_SIZE, LR = 3, 8, 5e-5

@torch.no_grad()
def evaluate_split(model, tokenizer, split_name):
    ds = datasets_[split_name]
    feats = ds.map(lambda x: preprocess_eval(x, tokenizer),
                   batched=True, remove_columns=ds.column_names)
    tensor_feats = feats.remove_columns(["example_id", "offset_mapping"])
    tensor_feats.set_format("torch")
    loader = DataLoader(tensor_feats, batch_size=BATCH_SIZE,
                        collate_fn=DataCollatorWithPadding(tokenizer))

    model.eval()
    starts, ends = [], []
    for batch in loader:
        out = model(**{k: v.to(DEVICE) for k, v in batch.items()})
        starts.append(out.start_logits.cpu().numpy())
        ends.append(out.end_logits.cpu().numpy())
    return postprocess(np.concatenate(starts), np.concatenate(ends), feats, ds)


def train_one(checkpoint, seed):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForQuestionAnswering.from_pretrained(checkpoint).to(DEVICE)

    train_ds = datasets_["train"]
    train_feats = train_ds.map(lambda x: preprocess_train(x, tokenizer),
                               batched=True, remove_columns=train_ds.column_names)
    train_feats.set_format("torch")
    loader = DataLoader(train_feats, shuffle=True, batch_size=BATCH_SIZE,
                        collate_fn=DataCollatorWithPadding(tokenizer))

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                              num_training_steps=NUM_EPOCHS * len(loader))

    best_val_f1, best_state = -1.0, None
    for epoch in range(NUM_EPOCHS):
        model.train()
        for batch in loader:
            loss = model(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
            loss.backward()
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        val = evaluate_split(model, tokenizer, "validation")
        print(f"  [{checkpoint.split('/')[-1]} seed={seed}] epoch {epoch+1} "
              f"val EM {val['exact_match']:.2f} F1 {val['f1']:.2f}")
        if val["f1"] > best_val_f1:                    # select on validation only
            best_val_f1 = val["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)                  # restore best-by-validation
    test = evaluate_split(model, tokenizer, "test")    # scored exactly once
    print(f"  -> HELD-OUT TEST  EM {test['exact_match']:.2f}  F1 {test['f1']:.2f}")
    return model, tokenizer, {"val_f1": best_val_f1, **test}

## 6. Run the comparison

Three seeds per model. Reporting mean and spread is what lets a reader tell a
real difference from run-to-run noise — with ~840 training questions, the spread
is expected to be wide.

In [ ]:
MODELS = {
    "romanian":     "dumitrescustefan/bert-base-romanian-cased-v1",
    "multilingual": "bert-base-multilingual-cased",
}
SEEDS = [42, 1337, 2024]

results, artifacts = collections.defaultdict(list), {}
for name, checkpoint in MODELS.items():
    print(f"\n=== {name} ({checkpoint}) ===")
    for seed in SEEDS:
        model, tokenizer, scores = train_one(checkpoint, seed)
        results[name].append(scores)
        if seed == SEEDS[0]:
            artifacts[name] = (model, tokenizer)   # keep the first seed for release

print(f"\n{'model':<14}{'test EM (mean±sd)':>22}{'test F1 (mean±sd)':>22}")
for name, runs in results.items():
    em = np.array([r["exact_match"] for r in runs])
    f1 = np.array([r["f1"] for r in runs])
    print(f"{name:<14}{em.mean():>13.2f} ± {em.std():<5.2f}{f1.mean():>13.2f} ± {f1.std():<5.2f}")

json.dump({k: v for k, v in results.items()}, open("heldout_results.json", "w"), indent=2)

## 7. Publish to the Hub

This step needs **your** token — run `notebook_login()` and paste it into the
widget. Never commit a token to the repository or share it with anyone.

Each card states the training data, the split methodology and the caveats, so the
numbers can't be mistaken for standard XQuAD benchmark results.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USER = "pop123ux"          # <- change if your username differs
PUSH = False                  # <- set True once you are happy with the results above

CARD = '''---
language: ro
license: mit
datasets: [xquad]
metrics: [exact_match, f1]
pipeline_tag: question-answering
tags: [question-answering, romanian, xquad, extractive-qa]
---

# {title}

`{base}` fine-tuned for Romanian extractive question answering on an
**article-level split** of [XQuAD-ro](https://github.com/google-deepmind/xquad).

## Results (held-out test split)

| Metric | Score |
|---|---:|
| Exact Match | {em:.2f} ± {em_sd:.2f} |
| F1 | {f1:.2f} ± {f1_sd:.2f} |

Mean ± standard deviation over seeds {seeds}. The checkpoint published here is
the seed-{first_seed} run, selected by validation F1.

## How it was trained

XQuAD ships no training split, so the ~1,190 questions were partitioned **by
article** (33 train / 7 validation / 8 test articles). Splitting by question
instead would put the same passage on both sides — 98.3% of test questions would
reuse a training context — so an article-level split is what makes these numbers
mean anything.

| | |
|---|---|
| Training questions | ~837 |
| Epochs | 3, AdamW, lr 5e-5, batch 8 |
| Max length / stride | 384 / 128 |
| Selection | best validation F1; test scored once |

## Limitations

- **The training set is tiny** (~837 questions). These scores reflect a very
  low-resource fine-tune, not what either model can reach with a proper corpus.
- **Not comparable to published XQuAD numbers**, which train on full SQuAD and
  use XQuAD purely for evaluation.
- **Domain is narrow** — XQuAD is translated SQuAD, so contexts are Wikipedia
  articles skewed towards American football.
- Trained as a learning project; not intended for production use.

Part of [HuggingFace-Project-Learning](https://github.com/{repo}).
'''

if PUSH:
    for name, (model, tokenizer) in artifacts.items():
        runs = results[name]
        em = np.array([r["exact_match"] for r in runs]); f1 = np.array([r["f1"] for r in runs])
        repo_id = f"{HF_USER}/bert-{name}-qa-xquad-ro"
        model.push_to_hub(repo_id); tokenizer.push_to_hub(repo_id)

        from huggingface_hub import ModelCard
        ModelCard(CARD.format(
            title=f"BERT ({name}) — Romanian extractive QA",
            base=MODELS[name], em=em.mean(), em_sd=em.std(), f1=f1.mean(), f1_sd=f1.std(),
            seeds=SEEDS, first_seed=SEEDS[0], repo="pop123-ux/HuggingFace-Project-Learning",
        )).push_to_hub(repo_id)
        print("pushed:", f"https://huggingface.co/{repo_id}")
else:
    print("PUSH is False — review the results above, then set PUSH = True to release.")

## What to do with the numbers

Whatever comes out, report it as what it is: **a held-out comparison on a very
small Romanian QA training set**, not a benchmark result. The earlier notebooks'
figures (EM 32.34 / 36.60) came from training and evaluating on the same file and
are not comparable to these — expect these to be *lower*, because the task is now
genuinely harder.

If the gap between the two models is smaller than the seed-to-seed spread, the
honest conclusion is "no measurable difference at this data scale", and that is a
perfectly good result to publish.